## Introducción

La tesoreria general de la republica entrega los datos que consolidan las patentes por no uso de derechos de aprovechamiento de aguas en Chile.
Este conjunto de datos permite clasificar las tesorerías regionales con más patentes por no uso de aprovechamiento de aguas además de permitir clasificar si los ddeudores son 
personas naturales o personas Juridicas.

**Objetivo general (Fase 2).** Construir un *pipeline* reproducible que deje el *dataset*
limpio, codificado y escalado, listo para el modelado posterior.

**Objetivos específicos.**
- Obtener y explorar los datos verificando su estructura y calidad.
- Limpiar gestionando rigurosamente los valores nulos.
- Transformar las variables categóricas nominales con codificación One-Hot.
- Estandarizar las variables continuas.
- Validar técnicamente el resultado.

**Atributos:** `id`, `gender`, `age`, `hypertension`, `heart_disease`, `ever_married`,
`work_type`, `Residence_type`, `avg_glucose_level`, `bmi`, `smoking_status` y la variable
objetivo `stroke` (1 = sufrió accidente cerebrovascular, 0 = no).

### Librerías utilizadas

| Librería | Para qué la usamos |
|---|---|
| `numpy` | Cálculo numérico y manejo de arreglos. |
| `pandas` | Cargar y manipular la tabla de datos (el `DataFrame`). |
| `matplotlib` | Generar los gráficos (*boxplots*). |
| `sklearn.preprocessing` | Las herramientas de codificación (`LabelEncoder`, `OneHotEncoder`) y escalamiento (`StandardScaler`). |

`np.random.seed(42)` fija la semilla aleatoria: garantiza que cualquier proceso con azar
dé **siempre el mismo resultado**, lo que asegura la *reproducibilidad* exigida en la fase.

In [ ]:
import sys                                   # intérprete del kernel
import subprocess                            # instalar paquetes faltantes
import importlib                             # refrescar el sistema de importación
import importlib.util                        # comprobar si un paquete está instalado

# Verificación del entorno: instala en el intérprete del kernel lo que falte.
REQUERIDOS = {                               # módulo: nombre del paquete en pip
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",               # se importa como sklearn, se instala como scikit-learn
}
faltantes = [pkg for mod, pkg in REQUERIDOS.items()
             if importlib.util.find_spec(mod) is None]

if faltantes:
    print("Instalando:", ", ".join(faltantes))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *faltantes])
    importlib.invalidate_caches()

import numpy as np                           # cálculo numérico y operaciones vectorizadas
import pandas as pd                          # estructuras tabulares: Series y DataFrame
import matplotlib.pyplot as plt              # gráficos de control
from sklearn import preprocessing            # LabelEncoder y OneHotEncoder
from sklearn.preprocessing import StandardScaler   # estandarización z-score

# Reproducibilidad: fijamos la semilla aleatoria del entorno
np.random.seed(42)

%matplotlib inline

In [ ]:
from pathlib import Path      # manejo de rutas independiente del sistema operativo
import sys

# sys.version trae la versión completa con fecha de compilación;
# split()[0] deja solo el número, que es lo único que hay que comprobar.
print("Python:", sys.version.split()[0])
for lib, mod in [("numpy", np), ("pandas", pd)]:
    print(f"{lib:8}:", mod.__version__)

# Estructura del proyecto. parents=True crea las carpetas intermedias;
# exist_ok=True evita el error si ya existen.
def localizar_raiz(inicio=None):
    """
    Devuelve la raíz del proyecto.

    Jupyter abre el kernel en la carpeta donde está guardado el cuaderno, no en la
    carpeta desde la que se lanzó JupyterLab. Como cada cuaderno vive en la carpeta
    de su fase (F1, F2, F3 o F4), si la carpeta de trabajo es una de ellas la raíz
    es la carpeta superior. Si el cuaderno se ejecuta desde la raíz, se usa tal cual.
    """
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio


RAIZ = localizar_raiz()                 # la misma función del cuaderno de la Fase 1
DIR_CRUDO = RAIZ / "data" / "raw"            # datos originales: nunca se modifican
DIR_PROCESADO = RAIZ / "data" / "processed"  # resultado del pipeline
DIR_DOCS = RAIZ / "docs"                     # diccionario, bitácora y metadatos
for carpeta in (DIR_CRUDO, DIR_PROCESADO, DIR_DOCS):
    carpeta.mkdir(parents=True, exist_ok=True)

ARCHIVO = DIR_CRUDO / "tb_patentes_aguas_december_2025_data.csv"
print("\nArchivo esperado en:", ARCHIVO)

### Procedencia del conjunto de datos

| Campo | Valor |
| --- | --- |
| Título | consolidacion de patentes por no uso de derechos de aprovechamiento de aguas |
| Autor | Tesoreía General de La República |
| Plataforma | Datos.gob.cl |
| Enlace | `https://datos.gob.cl/dataset/tb-patentes-aguas/resource/4d1d615f-5935-4760-b399-1dd76b281715` |
| Estructura esperada | 46.468 filas × 21 columnas originales· 34 nulos en `tesoreria_regional` |
| Unidad de observación | Pago patentes de agua|

**Referencia en APA 7 para el informe:**

> **Referencia en APA 7 para el informe:**
Tesorería General de la República. (2025). Reporte consolidado de recaudación y cartera morosa de patentes de aguas (diciembre 2025) [Conjunto de datos]. Ministerio de Hacienda, Gobierno de Chile. https://datos.gob.cl/dataset/tb-patentes-aguas/resource/4d1d615f-5935-4760-b399-1dd76b281715

Documentar la procedencia no es un trámite: sin ella, nadie puede verificar sobre
qué versión del archivo se trabajó.

## 1. Obtención de los datos

In [ ]:
def cargar_datos(ruta):
    """
    Carga un conjunto de datos desde un archivo CSV.

    Parámetros
    ----------
    ruta : str o Path
        Ruta del archivo CSV.

    Retorna
    -------
    pd.DataFrame
        Datos cargados en un DataFrame.

    Lanza
    -----
    FileNotFoundError
        Si el archivo no existe, con un mensaje que indica dónde debe estar.
    """
    ruta = Path(ruta)
    try:
        # En este archivo los faltantes de bmi vienen como el texto 'N/A':
        # na_values los convierte en NaN al leer.
        df = pd.read_csv(ruta, na_values=["N/A"])
    except FileNotFoundError:
        # from None oculta la traza interna de pandas: queda solo el mensaje útil.
        raise FileNotFoundError(
            f"No se encontró {ruta.name} en {ruta.parent.resolve()}.\n"
            "Descárguelo desde el enlace de la tabla de procedencia y guárdelo en "
            "data/raw/ de la raíz del proyecto."
        ) from None
    print(f"Datos cargados: {df.shape[0]} filas y {df.shape[1]} columnas.")
    return df


df_crudo = cargar_datos(ARCHIVO)

# Se conserva el original sin modificaciones y se trabaja sobre una copia
df = df_crudo.copy()
df.head()


## 2. Exploración inicial

**Qué hace este paso.** Antes de modificar nada, miramos el estado original de los datos.
Esto nos da la "línea base" contra la cual verificaremos después cada transformación.

In [ ]:
def explorar_dataframe(df):
    """Resumen exploratorio: dimensiones, tipos, nulos y estadisticos descriptivos."""
    print("Dimensiones:", df.shape)
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nValores nulos por columna:")
    print(df.isnull().sum())
    print("\nEstadisticos descriptivos (variables numericas):")
    return df.describe()


explorar_dataframe(df)

In [ ]:
# Conteo de categorias en las variables nominales (control de calidad de entrada)
columnas_categoricas = ['CUENTA', 'FORM_FOLIO', 'SALDO',
                        'PAGADO', 'MOV_TIPO']
for col in columnas_categoricas:
    print(f"\n{col}:")
    # dropna=False incluye los nulos como una categoría más: si una variable
    # tiene faltantes queremos verlos aquí, no que desaparezcan del conteo.
    print(df[col].value_counts(dropna=False))

### El diccionario de variables

Antes de limpiar hay que declarar **qué es cada variable**. Esta es la decisión
que determina todo el preprocesamiento posterior.

La idea central: **el tipo de dato no dice qué es una variable**. En este
conjunto `hypertension`, `id` y `age` son todas numéricas, y sin embargo exigen
tratamientos completamente distintos.

| Rol analítico | Qué preprocesamiento exige |
| --- | --- |
| **Continua** | Detección de atípicos, imputación, escalamiento |
| **Binaria** | Codificación 0/1 y revisión de desbalance |
| **Nominal** | *One-hot encoding* |
| **Ordinal** | Codificación con el orden declarado explícitamente |
| **Identificador** | Verificar unicidad y excluir del análisis |
| **Objetivo** | Se preserva sin transformar en esta fase |

In [ ]:
# El diccionario se DECLARA con lo que se sabe del dominio y se COMPLETA con lo
# observado en el archivo: no se afirma nada que no se verifique.
DECLARADO = [
    ("CUENTA", "identificador", "Identificador único de la cuenta"),
    ("RUT_ROL", "identificador", "RUT del titular, sin dígito verificador (dato personal: no se publica)"),
    ("TIPO_PERSONA", "binaria", "Tipo de persona registrado"),
    ("RUT_ROL_DV", "nominal", "Dígito verificador del RUT"),
    ("FORM_FOLIO", "continua", "Número de folio del formulario"),
    ("MONTO", "continua", "Monto asociado a la patente"),
    ("SALDO", "continua", "Saldo registrado de la patente"),
    ("PAGADO", "continua", "Monto pagado"),
    ("FECHA", "fecha", "Fecha del registro"),
    ("MOV_TIPO", "discreta", "Código del tipo de movimiento"),
    ("TESORERIA", "alta cardinalidad", "Tesorería asociada al registro"),
    ("TESORERIA_REGIONAL", "alta cardinalidad", "Tesorería regional correspondiente"),
    ("COMUNA", "alta cardinalidad", "Comuna asociada a la patente"),
    ("CONCESION", "alta cardinalidad", "Identificación de la concesión"),
    ("PROVINCIA", "alta cardinalidad", "Provincia asociada al registro"),
    ("OFICINA_CBR", "alta cardinalidad", "Oficina del Conservador de Bienes Raíces"),
    ("N_DE_CORRELATIVO_O_PATENTE", "continua", "Número correlativo o de patente"),
    ("ANIO_PATENTE", "discreta", "Año correspondiente a la patente"),
    ("UTM", "continua", "Valor registrado en UTM"),
    ("OFICINA", "alta cardinalidad", "Oficina asociada al registro"),
    ("FECHA_EJECUCION", "fecha", "Fecha de ejecución del proceso"),
]
diccionario = pd.DataFrame(DECLARADO, columns=["variable", "rol_analitico", "descripcion"])

# Columnas OBSERVADAS: se leen del archivo, no se escriben a mano
diccionario["dtype"] = [str(df_crudo[v].dtype) for v in diccionario["variable"]]
# nunique() cuenta valores distintos: distingue una binaria de una continua
diccionario["n_unicos"] = [int(df_crudo[v].nunique()) for v in diccionario["variable"]]
# Sobre booleanos, mean() devuelve la PROPORCIÓN de True: por 100 da el porcentaje
diccionario["pct_nulos"] = [round(df_crudo[v].isna().mean() * 100, 1)
                            for v in diccionario["variable"]]

# El diccionario es un entregable, no una tabla de paso: se guarda. Reemplaza al de la
# Fase 1 a propósito: conserva sus columnas y le agrega lo observado en el archivo.
diccionario.to_csv(DIR_DOCS / "diccionario_variables.csv", index=False)
diccionario

### Valores atípicos: medir antes de decidir

La sección siguiente imputa con la mediana «porque hay valores extremos». Esa
afirmación hay que **demostrarla**, no enunciarla. Se usa el criterio del rango
intercuartílico:

$$\text{atípico si} \quad x < Q_1 - 1{,}5 \cdot \text{RIC} \quad \text{o} \quad x > Q_3 + 1{,}5 \cdot \text{RIC}$$

In [ ]:
def detectar_atipicos_iqr(serie, factor=1.5):
    """Identifica valores atípicos por el criterio del rango intercuartílico.

    Retorna
    -------
    tuple(np.ndarray, float, float)
        Máscara booleana de atípicos, límite inferior y límite superior.
    """
    # dropna() es obligatorio: un NaN propagaría y devolvería NaN como cuartil
    valores = serie.dropna().to_numpy(dtype="float64")

    # np.percentile con una lista devuelve varios percentiles de una vez
    q1, q3 = np.percentile(valores, [25, 75])
    ric = q3 - q1                                  # rango intercuartílico
    limite_inf, limite_sup = q1 - factor * ric, q3 + factor * ric

    # El operador | es el "o" elemento a elemento de pandas. Con 'or' fallaría:
    # 'or' espera un único booleano, no una serie completa.
    mascara = (serie < limite_inf) | (serie > limite_sup)
    # Comparar con NaN da False pero deja NA: fillna(False) lo hace explícito
    return mascara.fillna(False).to_numpy(), limite_inf, limite_sup


filas = []
for col in ["MONTO", "SALDO", "PAGADO"]:
    mascara, inf, sup = detectar_atipicos_iqr(df[col])
    filas.append({
        "variable": col,
        "limite_inf": round(inf, 2),
        "limite_sup": round(sup, 2),
        "n_atipicos": int(mascara.sum()),          # sobre booleanos, sum() cuenta True
        "pct_atipicos": round(mascara.mean() * 100, 2),
        "media": round(df[col].mean(), 2),
        "mediana": round(df[col].median(), 2),
    })
pd.DataFrame(filas)

## 3. Limpieza de datos

Limpiar significa dejar los datos completos y consistentes. Tomamos dos decisiones, cada
una justificada técnicamente:

**a) Eliminar `id`.** Es un identificador único (un número distinto por paciente). No
contiene información que ayude a predecir el accidente cerebrovascular, así que lo
descartamos para que no introduzca ruido.

**b) Imputar `bmi` con la mediana.** `bmi` (índice de masa corporal) es la única columna
con valores faltantes. "Imputar" es rellenar esos huecos con un valor representativo.
Tenemos dos opciones:
- La **media** (promedio): se ve arrastrada por los valores extremos.
- La **mediana** (valor central): es **robusta** ante valores atípicos.

Como el `bmi` tiene valores atípicos (se ven en el *boxplot* del final), usar la mediana
evita que esos extremos sesguen el relleno. La función `imputar_nulos_numericos` permite
elegir la estrategia con un parámetro e informa cuántos nulos rellenó y con qué valor
(trazabilidad).

In [ ]:
def imputar_nulos_numericos(df, columna, estrategia="mediana"):
    """
    Imputa los valores nulos de una columna numerica.

    Parametros
    ----------
    df : pd.DataFrame
    columna : str
        Columna numerica a imputar.
    estrategia : str
        'media' o 'mediana'.

    Retorna
    -------
    pd.DataFrame
        DataFrame con la columna imputada.
    """
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame.")
    # isnull() da True/False por celda; sum() cuenta los True.
    # Se calcula ANTES de imputar: después ya no habría nulos que contar.
    n_nulos = int(df[columna].isnull().sum())
    if estrategia == "media":
        valor = df[columna].mean()
    elif estrategia == "mediana":
        valor = df[columna].median()
    else:
        raise ValueError("estrategia debe ser 'media' o 'mediana'")
    df = df.copy()
    df[columna] = df[columna].fillna(valor)
    print(f"'{columna}': {n_nulos} nulos imputados con la {estrategia} = {valor:.2f}")
    return df

In [ ]:
df.isnull().sum()

In [ ]:
# Antes de descartarlo se comprueba que 'id' sea efectivamente un identificador:
# is_unique devuelve True si no hay valores repetidos.
#assert df['RUT_ROL'].is_unique, "'RUT_ROL' tiene valores repetidos: revísalo antes de descartarlo"

# drop(columns=[...]) devuelve un DataFrame NUEVO sin esa columna.
df = df.drop(columns=['TESORERIA_REGIONAL'])
df.head()

Imputamos los nulos de `SALDO` con la mediana:

Debido a que la columna que se analiza "TESORERIA_REGIONAL" no contiene numeros (pero si nulos), esto no aplica en nuestro caso

In [ ]:
df = imputar_nulos_numericos(df, "SALDO", estrategia="mediana")

### Comparar los métodos antes de elegir

La celda anterior aplica la mediana. Falta lo que la rúbrica evalúa: **por qué
esa y no otra**. Ninguna estrategia es mejor en abstracto.

| Método | Cuándo conviene | Qué distorsiona |
| --- | --- | --- |
| **Eliminar filas** | Faltantes escasos y aparentemente aleatorios | Pierde muestra; sesga si no son aleatorios |
| **Media** | Distribución aproximadamente simétrica | La arrastran los extremos; reduce la dispersión |
| **Mediana** | Hay valores extremos o asimetría | Reduce la dispersión; ignora las demás variables |
| **Mediana por grupo** | El valor depende de otra variable observada | Reduce la dispersión dentro de cada grupo |

En este problema esto no es necesario ya que no hay nulos

In [ ]:
def comparar_imputaciones(datos, columna, agrupador=None, n_grupos=4):
    """Compara el efecto de cuatro estrategias de imputación sobre una variable.

    Retorna
    -------
    pd.DataFrame con el efecto de cada estrategia sobre la distribución.
    """
    original = datos[columna]
    if original.isna().sum() == 0:            # sin faltantes no hay nada que comparar
        raise ValueError(f"'{columna}' no tiene valores faltantes.")

    resultados = {
        "sin imputar (referencia)": original.dropna(),
        "eliminar filas": original.dropna(),
        "media": original.fillna(original.mean()),
        "mediana": original.fillna(original.median()),
    }
    if agrupador is not None:
        # qcut corta en tramos con la MISMA cantidad de casos (cuartiles si q=4);
        # duplicates="drop" evita el error cuando dos cortes coinciden.
        tramos = pd.qcut(datos[agrupador], q=n_grupos, duplicates="drop")
        # transform("median") devuelve una serie del MISMO largo, con la mediana
        # del grupo en cada fila. Con agg() se obtendría una fila por grupo,
        # que no sirve para rellenar.
        resultados["mediana por grupo"] = original.fillna(
            original.groupby(tramos, observed=True).transform("median"))

    filas = [{
        "estrategia": nombre,
        "n": len(serie.dropna()),
        "media": round(serie.mean(), 3),
        "desv_est": round(serie.std(), 3),
        "asimetria": round(serie.skew(), 3),     # skew() mide la cola de la distribución
    } for nombre, serie in resultados.items()]

    tabla = pd.DataFrame(filas)
    referencia = tabla.iloc[0]["desv_est"]       # iloc[0] es la fila de referencia
    # La columna decisiva: cuánto deforma cada método la dispersión original
    tabla["cambio_desv_%"] = ((tabla["desv_est"] - referencia) / referencia * 100).round(2)
    tabla["filas_perdidas"] = len(datos) - tabla["n"]
    return tabla


comparar_imputaciones(df_crudo, "SALDO", agrupador="TESORERIA_REGIONAL")

**Cómo se lee.** `cambio_desv_%` es la columna que decide: toda imputación por
un valor central concentra los datos y **reduce artificialmente la desviación
estándar**. El método que menos la reduce es el que menos deforma la variable.
`filas_perdidas` muestra el costo de eliminar, y `asimetria` indica si la media
representa bien el centro.

> **Si las filas dan casi lo mismo**, la variable es simétrica y las estrategias
> son equivalentes: lo correcto es declararlo y elegir la más simple. Escribir
> «se descarta la media por la asimetría» cuando la tabla muestra asimetría nula
> es exactamente la incoherencia que la rúbrica penaliza.

In [ ]:
# La bandera de imputación: que un valor faltara puede ser informativo en sí
# mismo, de modo que el hecho se conserva en una columna aparte.
df["bmi_imputado"] = df_crudo["bmi"].isna().astype(int)   # 1 = valor imputado
print(f"Filas marcadas como imputadas: {int(df['bmi_imputado'].sum())}")
print(f"Desviación estándar de 'bmi': "
      f"{df_crudo['bmi'].std():.3f} -> {df['bmi'].std():.3f}")

Se pasa al punto 4, ya que no es necesario limpiar las columnas en analisis

## 4. Transformación: codificación One-Hot
Los modelos solo entienden números, no texto. Por eso hay que convertir las variables de
texto en números. Pero **cómo** las convertimos importa:



Por ejemplo, la columna `TESORERIA_REGIONAL` se convierte en dos columnas:

| Residence_type | → | res_Rural | res_Urban |
|---|---|---|---|
| Urban | | 0 | 1 |
| Rural | | 1 | 0 |

Mantenemos el método visto en el curso —**`LabelEncoder` seguido de `OneHotEncoder`**— pero
lo encapsulamos en una sola función para no repetir el mismo bloque cinco veces.